## 7. Conclusiones

### Rendimiento de los metodos
- **SFS (Sequential Forward Selection)** obtuvo el mejor accuracy (78.35%) con 9 features, superando al modelo base con la mitad de las variables.
- **SBS (Sequential Backward Selection)** logro un accuracy competitivo (78.21%) con solo 7 features, el subconjunto mas compacto.
- **RFE (Recursive Feature Elimination)** mantuvo un accuracy similar al base (76.79%) con 8 features.

### Variables clave para predecir obesidad
Las features consistentemente seleccionadas por los tres metodos son las mas robustas y representan los factores mas determinantes de la obesidad:
- **FAVC** (consumo frecuente de alimentos hipercaloricos)
- **CAEC** (consumo entre comidas)
- **SCC** (monitoreo de calorias)
- **family_history_with_overweight** (antecedentes familiares)
- **Age** (edad)

### Insight principal
Los tres metodos coinciden en que los **habitos alimenticios** (FAVC, CAEC, SCC) y los **antecedentes familiares** son mas predictivos que factores como el transporte o el consumo de agua. Esto sugiere que las intervenciones de salud publica deberian enfocarse en la educacion alimentaria y el monitoreo calorico.

### Recomendacion
Para produccion, se recomienda **SBS con 7 features** por ofrecer el mejor balance entre parsimonia y rendimiento. Un modelo mas simple es mas interpretable, requiere menos datos para entrenamiento y es menos propenso al sobreajuste.

---
*Proyecto de feature selection comparando tres metodos wrapper sobre datos de habitos alimenticios y obesidad.*

# Seleccion de Caracteristicas con Metodos Wrapper para Prediccion de Obesidad

En este proyecto se implementan y comparan tres metodos de seleccion de caracteristicas tipo **wrapper** para optimizar un modelo de regresion logistica que predice obesidad a partir de habitos alimenticios y condicion fisica.

**Metodos evaluados:**
- **Sequential Forward Selection (SFS):** Agrega variables una a una, conservando las que mejoran el rendimiento
- **Sequential Backward Selection (SBS):** Elimina variables una a una, removiendo las menos utiles
- **Recursive Feature Elimination (RFE):** Elimina recursivamente las variables con menor importancia segun el modelo

**Dataset:** Encuesta sobre habitos alimenticios y peso corporal del [UCI Machine Learning Repository](https://archive.ics.uci.edu/ml/datasets/Estimation+of+obesity+levels+based+on+eating+habits+and+physical+condition+), con 18 variables predictoras y una variable objetivo binaria (obeso/no obeso).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler
from sklearn.feature_selection import RFE
from mlxtend.feature_selection import SequentialFeatureSelector as SFS
from mlxtend.plotting import plot_sequential_feature_selection as plot_sfs

plt.style.use('seaborn-v0_8-whitegrid')
plt.rcParams['figure.figsize'] = (12, 7)
plt.rcParams['font.size'] = 12

print("Librerias cargadas correctamente")

## 1. Carga y Exploracion del Dataset

El dataset contiene 18 variables predictoras recopiladas a traves de una encuesta sobre habitos alimenticios y condicion fisica. Las variables incluyen informacion demografica (genero, edad), habitos alimenticios (frecuencia de comidas, consumo de vegetales, alcohol), actividad fisica y medios de transporte. La variable objetivo `NObeyesdad` indica si la persona es obesa (1) o no (0).

In [ ]:
obesity = pd.read_csv("obesity.csv")
print(f"Dataset: {obesity.shape[0]} registros, {obesity.shape[1]} variables")
print(f"Distribucion de la variable objetivo:")
print(f"  No obeso (0): {(obesity['NObeyesdad']==0).sum()} ({(obesity['NObeyesdad']==0).mean()*100:.1f}%)")
print(f"  Obeso (1):    {(obesity['NObeyesdad']==1).sum()} ({(obesity['NObeyesdad']==1).mean()*100:.1f}%)")
print()
obesity.head()

## 2. Modelo Base: Regresion Logistica con Todas las Variables

Primero se entrena un modelo de regresion logistica utilizando las 18 variables disponibles. Este servira como **linea base** para comparar el rendimiento de los metodos de seleccion de caracteristicas.

In [ ]:
# Separar variables predictoras y objetivo
X = obesity.drop(["NObeyesdad"], axis=1)
y = obesity["NObeyesdad"]

print(f"Variables predictoras: {X.shape[1]}")
print(f"Columnas: {list(X.columns)}")

### Entrenamiento y accuracy base

In [ ]:
# Modelo base con todas las variables
lr = LogisticRegression(max_iter=1000)
lr.fit(X, y)

lr_score_base = lr.score(X, y)
print(f"Accuracy del modelo base (18 variables): {lr_score_base:.4f} ({lr_score_base*100:.2f}%)")

## 3. Metodo 1: Sequential Forward Selection (SFS)

El SFS comienza con un conjunto vacio de features y en cada iteracion agrega la variable que produce la mayor mejora en el rendimiento del modelo. Se seleccionan hasta **9 features**.

In [ ]:
sfs = SFS(lr, k_features=9, forward=True, floating=False, scoring='accuracy', cv=0)
sfs.fit(X, y)

sfs_features = list(sfs.subsets_[9]['feature_names'])
sfs_score = sfs.subsets_[9]['avg_score']

print(f"Features seleccionadas ({len(sfs_features)}):")
for i, f in enumerate(sfs_features, 1):
    print(f"  {i}. {f}")
print(f"\nAccuracy con SFS: {sfs_score:.4f} ({sfs_score*100:.2f}%)")
print(f"Mejora vs base: {(sfs_score - lr_score_base)*100:+.2f} puntos porcentuales")

### Evolucion del accuracy por numero de features (SFS)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_sfs(sfs.get_metric_dict(), ax=ax)
ax.set_title('Sequential Forward Selection - Accuracy vs Numero de Features', fontsize=14, fontweight='bold')
ax.set_xlabel('Numero de Features', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.axhline(y=lr_score_base, color='red', linestyle='--', alpha=0.7, label=f'Base (18 vars): {lr_score_base:.4f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("Se observa que con 9 features se supera la accuracy del modelo base de 18 variables.")

## 4. Metodo 2: Sequential Backward Selection (SBS)

A diferencia del SFS, el SBS comienza con todas las variables y elimina una en cada iteracion, removiendo la que menos contribuye al rendimiento. Se reduce a **7 features**.

In [ ]:
sbs = SFS(lr, k_features=7, forward=False, floating=False, scoring='accuracy', cv=0)
sbs.fit(X, y)

sbs_features = list(sbs.subsets_[7]['feature_names'])
sbs_score = sbs.subsets_[7]['avg_score']

print(f"Features seleccionadas ({len(sbs_features)}):")
for i, f in enumerate(sbs_features, 1):
    print(f"  {i}. {f}")
print(f"\nAccuracy con SBS: {sbs_score:.4f} ({sbs_score*100:.2f}%)")
print(f"Mejora vs base: {(sbs_score - lr_score_base)*100:+.2f} puntos porcentuales")

### Evolucion del accuracy por numero de features (SBS)

In [ ]:
fig, ax = plt.subplots(figsize=(12, 6))
plot_sfs(sbs.get_metric_dict(), ax=ax)
ax.set_title('Sequential Backward Selection - Accuracy vs Numero de Features', fontsize=14, fontweight='bold')
ax.set_xlabel('Numero de Features', fontsize=12)
ax.set_ylabel('Accuracy', fontsize=12)
ax.axhline(y=lr_score_base, color='red', linestyle='--', alpha=0.7, label=f'Base (18 vars): {lr_score_base:.4f}')
ax.legend(fontsize=11)
plt.tight_layout()
plt.show()

print("SBS logra mantener el rendimiento con solo 7 variables, eliminando 11 features redundantes.")

## 5. Metodo 3: Recursive Feature Elimination (RFE)

RFE entrena el modelo, evalua la importancia de cada feature (basandose en los coeficientes del modelo), elimina la menos importante y repite el proceso. Requiere **estandarizacion previa** para que los coeficientes sean comparables. Se seleccionan **8 features**.

In [ ]:
# Guardar nombres de features antes de estandarizar
features = X.columns

# Estandarizar los datos (necesario para que RFE compare coeficientes correctamente)
scaler = StandardScaler()
X_scaled = pd.DataFrame(scaler.fit_transform(X), columns=features)

# Aplicar RFE
rfe = RFE(lr, n_features_to_select=8)
rfe.fit(X_scaled, y)

rfe_features = [f for f, support in zip(features, rfe.support_) if support]
rfe_score = rfe.score(X_scaled, y)

print(f"Features seleccionadas ({len(rfe_features)}):")
for i, f in enumerate(rfe_features, 1):
    print(f"  {i}. {f}")
print(f"\nAccuracy con RFE: {rfe_score:.4f} ({rfe_score*100:.2f}%)")
print(f"Mejora vs base: {(rfe_score - lr_score_base)*100:+.2f} puntos porcentuales")

## 6. Comparacion de los Tres Metodos

Ahora se comparan los resultados de los tres metodos de seleccion para identificar patrones y determinar cual produce el mejor balance entre rendimiento y parsimonia.

In [ ]:
# Tabla comparativa de resultados
results = pd.DataFrame({
    'Metodo': ['Base (todas)', 'SFS (Forward)', 'SBS (Backward)', 'RFE'],
    'Num Features': [18, len(sfs_features), len(sbs_features), len(rfe_features)],
    'Accuracy': [lr_score_base, sfs_score, sbs_score, rfe_score]
})
results['Diferencia vs Base'] = results['Accuracy'] - lr_score_base
results['Accuracy %'] = results['Accuracy'].apply(lambda x: f"{x*100:.2f}%")

print("=" * 65)
print("COMPARACION DE METODOS DE SELECCION DE FEATURES")
print("=" * 65)
for _, row in results.iterrows():
    diff = f"{row['Diferencia vs Base']*100:+.2f}pp" if row['Metodo'] != 'Base (todas)' else "---"
    print(f"  {row['Metodo']:<18} | {int(row['Num Features']):>2} features | Accuracy: {row['Accuracy %']:>7} | {diff}")
print("=" * 65)

In [ ]:
# Visualizacion comparativa
fig, axes = plt.subplots(1, 2, figsize=(16, 6))

# Grafico de accuracy por metodo
colors = ['#95a5a6', '#3498db', '#2ecc71', '#e74c3c']
bars = axes[0].bar(results['Metodo'], results['Accuracy'], color=colors, edgecolor='white', width=0.6)
axes[0].set_ylim(0.74, 0.80)
axes[0].set_title('Accuracy por Metodo de Seleccion', fontsize=14, fontweight='bold')
axes[0].set_ylabel('Accuracy', fontsize=12)
for bar, val in zip(bars, results['Accuracy']):
    axes[0].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.001, 
                f'{val:.4f}', ha='center', va='bottom', fontsize=11, fontweight='bold')
axes[0].tick_params(axis='x', rotation=15)

# Features seleccionadas por cada metodo (heatmap)
all_features_set = sorted(set(sfs_features + sbs_features + rfe_features))
selection_matrix = pd.DataFrame({
    'SFS': [1 if f in sfs_features else 0 for f in all_features_set],
    'SBS': [1 if f in sbs_features else 0 for f in all_features_set],
    'RFE': [1 if f in rfe_features else 0 for f in all_features_set]
}, index=all_features_set)

sns.heatmap(selection_matrix, annot=True, cmap='YlGnBu', cbar=False, ax=axes[1],
            linewidths=0.5, fmt='d', square=False)
axes[1].set_title('Features Seleccionadas por Metodo', fontsize=14, fontweight='bold')
axes[1].set_xlabel('Metodo', fontsize=12)

plt.tight_layout()
plt.show()

# Features consistentes
common_features = set(sfs_features) & set(sbs_features) & set(rfe_features)
print(f"\nFeatures seleccionadas por los 3 metodos ({len(common_features)}):")
for f in sorted(common_features):
    print(f"  - {f}")